In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

spark = SparkSession.builder \
    .appName("FraudPlatform-Phase2") \
    .master("local[2]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.default.parallelism", "2") \
    .config("spark.driver.maxResultSize", "512m") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")   # reduces noise
print("Spark session created successfully")
print(f"Spark version: {spark.version}")

KeyboardInterrupt: 

In [ ]:
import os

raw_path = "../data/raw/Paysim_dataset.csv"
parquet_path = "../data/parquet/transactions"

# Only convert once
if not os.path.exists(parquet_path):
    print("Converting CSV → Parquet (this will take a few minutes)...")
    
    df_raw = spark.read.csv(
        raw_path,
        header=True,
        inferSchema=True
    )
    
    # Write as Parquet with good partitioning
    df_raw.write \
        .mode("overwrite") \
        .parquet(parquet_path)
    
    print("Conversion completed!")
else:
    print("Parquet already exists. Skipping conversion.")

# Always read from Parquet going forward
df = spark.read.parquet(parquet_path)

print(f"Total rows: {df.count():,}")
df.printSchema()
df.show(5, truncate=False)

In [ ]:
print("=== Fraud Distribution ===")
df.groupBy("isFraud").count().show()

print("\n=== Transaction Types ===")
df.groupBy("type").count().orderBy(F.desc("count")).show()

print("\n=== Amount Statistics ===")
df.select(
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.avg("amount").alias("avg_amount"),
    F.expr("percentile(amount, 0.5)").alias("median_amount")
).show()

In [ ]:
print("Creating features with Window functions...")

# Define window for customer behaviour
customer_window = Window.partitionBy("nameOrig").orderBy("step")

df_feat = df.withColumn("log_amount", F.log1p("amount")) \
    .withColumn("orig_balance_diff", F.col("newbalanceOrig") - F.col("oldbalanceOrg")) \
    .withColumn("dest_balance_diff", F.col("newbalanceDest") - F.col("oldbalanceDest")) \
    .withColumn("orig_error", F.abs(F.col("oldbalanceOrg") - F.col("amount") - F.col("newbalanceOrig"))) \
    .withColumn("dest_error", F.abs(F.col("oldbalanceDest") + F.col("amount") - F.col("newbalanceDest"))) \
    .withColumn("is_orig_balance_zero", (F.col("oldbalanceOrg") == 0).cast("int")) \
    .withColumn("is_dest_balance_zero", (F.col("oldbalanceDest") == 0).cast("int")) \
    .withColumn("prev_amount", F.lag("amount", 1).over(customer_window)) \
    .withColumn("time_since_last_txn", F.col("step") - F.lag("step", 1).over(customer_window))

df_feat.select(
    "nameOrig", "step", "amount", "prev_amount", "time_since_last_txn", "isFraud"
).show(10)

In [ ]:
# 1. Check current partitions
print(f"Current partitions: {df_feat.rdd.getNumPartitions()}")

# 2. Repartition by customer (useful for customer-level features)
df_part = df_feat.repartition(8, "nameOrig")
print(f"After repartition by nameOrig: {df_part.rdd.getNumPartitions()}")

# 3. Create a small lookup table (simulate merchant risk)
merchant_risk = df.groupBy("nameDest").agg(
    F.avg("isFraud").alias("merchant_fraud_rate"),
    F.count("*").alias("merchant_txn_count")
).filter(F.col("merchant_txn_count") > 10)

# Broadcast join (important Big Data concept)
from pyspark.sql.functions import broadcast

df_enriched = df_part.join(
    broadcast(merchant_risk),
    on="nameDest",
    how="left"
)

df_enriched.select("nameOrig", "nameDest", "amount", "merchant_fraud_rate", "isFraud").show(5)

In [ ]:
# Caution: Only run this on a sample if memory is tight
sample_size = 300_000

print("Comparing Pandas vs Spark on same operation...")

# Spark timing
start = time.time()
spark_result = df.filter(F.col("amount") > 10000) \
                 .groupBy("type") \
                 .agg(F.avg("amount").alias("avg_amount"), F.count("*").alias("cnt")) \
                 .collect()
spark_time = time.time() - start
print(f"Spark time: {spark_time:.2f} seconds")

# Pandas timing (on sample)
pdf = df.limit(sample_size).toPandas()
start = time.time()
pandas_result = pdf[pdf["amount"] > 10000].groupby("type")["amount"].agg(["mean", "count"])
pandas_time = time.time() - start
print(f"Pandas time (on {sample_size:,} rows): {pandas_time:.2f} seconds")

In [ ]:
feature_path = "../data/parquet/transaction_features"

df_feat.write \
    .mode("overwrite") \
    .partitionBy("type") \
    .parquet(feature_path)

print("Feature dataset written successfully!")

In [ ]:
spark.stop()
print("Spark session stopped. Memory released.")